# Fetal Planes – Data Preparation

In [ ]:
import math
import os
import random
import re
import shutil
from collections.abc import Callable, Sequence
from itertools import combinations
from math import ceil
from multiprocessing import Pool, cpu_count
from typing import Any, Literal

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from ipywidgets import Video
from lightning import LightningDataModule, LightningModule
from PIL import Image
from skimage.metrics import structural_similarity
from sklearn.model_selection import GroupShuffleSplit
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, ConfusionMatrix, F1Score, MaxMetric, MeanMetric
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_numpy_images,
    show_pytorch_images,
)

data_dir = root / "data"

# Annotate FETAL_PLANES CSV

Enrich `data.csv` with three new columns — `Subset`, `Duplicate`, and `Valid` — and save the result as `data_fix.csv`.

## Assign Subsets

Split the dataset into `train`, `val`, and `test` subsets using a patient-level group split so that no patient appears in more than one subset.

### Load CSV

Load `FETAL_PLANES/data.csv` and keep the columns needed for splitting and labelling.

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data.csv")
fetal_planes = fetal_planes[["Image_name", "Patient_num", "Plane", "Brain_plane", "Train"]]
fetal_planes = fetal_planes.reset_index(drop=True)
fetal_planes

### Split into Train / Val / Test

Apply `GroupShuffleSplit` (seed 5724) on the training patients to carve out a ~10 % validation set.
All rows already marked `Train=0` become the held-out `test` subset.

In [ ]:
train_fetal_planes = fetal_planes[fetal_planes["Train"] == 1].reset_index(drop=True)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=5724)
split = splitter.split(list(range(len(train_fetal_planes))), groups=train_fetal_planes["Patient_num"])
train_idx, val_idx = next(split)
val_idx = set(val_idx)

fetal_planes["Subset"] = "test"

for idx, image_name in tqdm(enumerate(train_fetal_planes["Image_name"]), total=len(train_fetal_planes)):
    if idx in val_idx:
        subset = "val"
    else:
        subset = "train"
    fetal_planes.loc[fetal_planes["Image_name"] == image_name, "Subset"] = subset

fetal_planes

### Statistics

Distribution of plane classes, brain-plane labels, and subset sizes.
The final cell shows a pivot of brain-plane counts broken down by train/val/test.

In [ ]:
fetal_planes["Plane"].value_counts()

In [ ]:
fetal_planes["Brain_plane"].value_counts()

In [ ]:
fetal_planes["Subset"].value_counts()

In [ ]:
fetal_planes[fetal_planes["Plane"] == "Fetal brain"]["Subset"].value_counts()

In [ ]:
def get_subset_brain_planes_df(df, subset):
    if subset != "all":
        df = df[df["Subset"] == subset]
    df = df["Brain_plane"].value_counts().reset_index()
    df.columns = ["Brain_plane", "Count"]
    total_row = pd.DataFrame([{"Brain_plane": "Total", "Count": df["Count"].sum()}])
    df = pd.concat([df, total_row], ignore_index=True)
    df["Subset"] = subset
    return df


def get_brain_planes_count(df):
    all_df = get_subset_brain_planes_df(df, "all")
    train_df = get_subset_brain_planes_df(df, "train")
    val_df = get_subset_brain_planes_df(df, "val")
    test_df = get_subset_brain_planes_df(df, "test")
    df = pd.concat([all_df, train_df, val_df, test_df], ignore_index=True)
    df = df.pivot_table(index="Brain_plane", columns="Subset", values="Count", fill_value=0)
    df = df.reindex(columns=["train", "val", "test", "all"])
    df = df.reindex(["Trans-ventricular", "Trans-thalamic", "Trans-cerebellum", "Other", "Not A Brain", "Total"])
    df = df.astype(int)
    return df


get_brain_planes_count(fetal_planes)

### Save CSV

Persist the dataframe — now with the `Subset` column — as `data_fix.csv`.

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes

## Remove Duplicates

Detect near-duplicate images using pairwise SSIM within each patient's images.
Duplicates are chained into groups and recorded in a `Duplicate` column; they are excluded from training but not deleted.

### Load CSV

Reload `data_fix.csv` to start the deduplication pass.

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes

### Find Duplicates

For every patient, compare all image pairs with SSIM in parallel.
Pairs scoring above the threshold are recorded; each duplicate image gets the filename of its root original stored in the `Duplicate` column.

In [ ]:
def read_fetal_planes_image(image_name):
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    return read_grayscale_image(image_path)


def read_grayscale_image(image_path):
    image = Image.open(image_path).convert("L")
    image_array = np.asarray(image)
    return image_array


def calculate_ssim(image1, image2):
    if image1.shape != image2.shape:
        return 0.0

    ssim = structural_similarity(image1, image2, data_range=255)
    if ssim > 0.95:
        return ssim
    elif ssim > 0.5:
        image1_crop = image1[:, : int(image1.shape[1] * 0.75)]
        image2_crop = image2[:, : int(image1_crop.shape[1])]
        return structural_similarity(image1_crop, image2_crop, data_range=255)
    else:
        return ssim


def process_patient_data(patient_num):
    """
    The worker function to be run in a separate process.
    It processes all images for a single patient and returns the results.
    """
    # These lists will store the results for this patient only
    local_duplicates = []

    patient_df = fetal_planes[fetal_planes["Patient_num"] == patient_num]
    indexes = patient_df.index.to_list()

    # Read all images for the current patient
    images = {}
    for index in indexes:
        image = read_fetal_planes_image(patient_df.loc[index, "Image_name"])
        images[index] = image

    # Pairwise comparison of all images for the patient
    duplicate_indexes: set[int] = set()
    indexes_pairs = list(combinations(indexes, 2))
    for index1, index2 in indexes_pairs:
        # Skip images already linked to an earlier duplicate for this patient
        if index2 in duplicate_indexes:
            continue

        image1 = images[index1]
        image2 = images[index2]
        ssim = calculate_ssim(image1, image2)

        if ssim > 0.95:
            local_duplicates.append((index1, index2, ssim))
            duplicate_indexes.add(index2)

    return local_duplicates


fetal_planes["Duplicate"] = None

num_processes = cpu_count()
duplicates = []

print(f"Starting parallel processing with {num_processes} cores...")
with Pool(num_processes) as pool:
    patients = fetal_planes["Patient_num"].unique().tolist()
    for local_duplicates in tqdm(pool.imap(process_patient_data, patients), total=len(patients), desc="Patients"):
        duplicates.extend(local_duplicates)

for base_index, duplicate_index, _ in duplicates:
    fetal_planes.loc[duplicate_index, "Duplicate"] = int(base_index)

print("\nProcessing complete.")
print(f"Found {len(duplicates)} similar images.")  # 1072

### Group Duplicates

Walk the duplicate links to form connected chains so that every member of a duplicate cluster points to the same root image.

In [ ]:
images = []

root_duplicates = []
rows_visited = set()
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    if index in rows_visited:
        continue

    indexes = [index]
    duplicate_indexes = []
    while len(indexes) > 0:
        current_index = indexes.pop(0)
        rows_visited.add(current_index)
        duplicate_indexes.append(current_index)
        indexes.extend(fetal_planes[fetal_planes["Duplicate"] == current_index].index.to_list())

    if len(duplicate_indexes) > 1:
        root_duplicates.append(duplicate_indexes)

print(len(root_duplicates))

### Inspect Duplicate Groups

Display four sample duplicate groups (indices 0, 60, 500, 600) with SSIM scores to confirm the detection is working correctly.

In [ ]:
images = []

for root_duplicate in tqdm([root_duplicates[i] for i in [0, 60, 500, 600]]):
    base_image = None
    for index in root_duplicate:
        image = read_fetal_planes_image(fetal_planes.Image_name[index])

        if base_image is None:
            base_image = image
            label = f"{index}"
        else:
            ssim = calculate_ssim(base_image, image)
            label = f"{index}: {ssim:.3f}"

        images.append((image, label))

        if len(images) >= 64:
            break
    if len(images) >= 64:
        break

print(len(images))
if images:
    show_numpy_images(images[:36])

### Statistics

Brain-plane counts in the full dataset vs. after excluding flagged duplicates.

In [ ]:
get_brain_planes_count(fetal_planes)

In [ ]:
get_brain_planes_count(fetal_planes[fetal_planes["Duplicate"].isna()])

### Save CSV

Save `data_fix.csv` with the new `Duplicate` column populated.

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes

## Remove Invalid Data

Manually curate the dataset by marking mislabelled images as invalid (`Valid=0`) and, where appropriate, reassigning their brain-plane label via `New_brain_plane`.

### Load CSV

Reload `data_fix.csv` and add two helper columns: `Valid` (default 1) and `New_brain_plane` (initially empty).

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes["Valid"] = 1
fetal_planes["New_brain_plane"] = None
fetal_planes

### Exclude Mislabelled Brain Images

A curated list of images that are labelled as a brain plane but contain no brain structure.
These are marked `Valid=0` and displayed for confirmation.

In [ ]:
images = []

remove_images = [
    "Patient00765_Plane3_5_of_6",
    "Patient00765_Plane3_6_of_6",
    "Patient00850_Plane3_4_of_4",
    "Patient01028_Plane3_3_of_3",
    # Other brain images
    "Patient00684_Plane3_2_of_3",
    "Patient00684_Plane3_3_of_3",
    "Patient00721_Plane3_2_of_3",
    "Patient01013_Plane3_4_of_4",
    "Patient01272_Plane3_3_of_3",
    "Patient01304_Plane3_4_of_6",
    "Patient01646_Plane3_3_of_3",
    "Patient00803_Plane3_1_of_5",
    # Axial brain images
    "Patient00914_Plane3_3_of_4",
    "Patient01280_Plane3_3_of_5",
    "Patient01635_Plane3_1_of_4",
]
for image_name in remove_images:
    for index, row in fetal_planes[fetal_planes.Image_name == image_name].iterrows():
        fetal_planes.loc[index, "Valid"] = 0
        # image = read_fetal_planes_image(image_name)
        # images.append((image, f"{index}: {image_name}"))
        # print(f"{index:<6}: {image_name} \t{row['Plane']} \t{row['Brain_plane']}")

if images:
    show_numpy_images(images[:36])

### Reclassify 'Not A Brain' Images Showing a Brain

Commented-out block of relabelling rules for images currently in the 'Not A Brain' class that actually show a recognisable brain structure.
Kept for reference; the active loop makes no changes.

In [ ]:
images = []

# Brain - TT, TC, TV
# Mid-sagittal - head profil
# Axial - top with nose

# (image name, new plane, valid)
remove_images = [
    ("Patient00168_Plane1_10_of_23", "Axial", 1),
    # ("Patient00216_Plane1_1_of_1", "Axial", 0), # abo (brzuch)
    ("Patient00720_Plane1_22_of_27", "Axial", 1),
    ("Patient00720_Plane1_27_of_27", "Axial", 1),
    ("Patient00720_Plane1_7_of_27", "Axial", 1),
    ("Patient00779_Plane1_1_of_1", "Axial", 1),
    ("Patient00791_Plane1_1_of_25", "Axial", 1),
    ("Patient00799_Plane1_21_of_28", "Axial", 1),
    ("Patient00830_Plane1_11_of_49", "Axial", 1),
    ("Patient00834_Plane1_12_of_34", "Axial", 1),
    ("Patient00890_Plane1_8_of_19", "Axial", 1),
    ("Patient00897_Plane1_1_of_1", "Axial", 1),
    ("Patient00897_Plane1_1_of_1", "Axial", 1),
    ("Patient00906_Plane1_7_of_27", "Axial", 1),
    ("Patient00914_Plane3_3_of_4", "Axial", 1),  # 1
    ("Patient00944_Plane1_1_of_1", "Axial", 1),
    ("Patient01003_Plane1_1_of_1", "Axial", 1),
    ("Patient01035_Plane1_1_of_1", "Axial", 1),
    ("Patient01162_Plane1_1_of_1", "Axial", 1),
    ("Patient01179_Plane1_1_of_1", "Axial", 1),
    ("Patient01179_Plane1_1_of_1", "Axial", 1),
    ("Patient01185_Plane1_9_of_15", "Axial", 1),
    ("Patient01280_Plane3_3_of_5", "Axial", 1),
    ("Patient01286_Plane1_1_of_1", "Axial", 1),
    ("Patient01327_Plane1_8_of_21", "Axial", 1),
    ("Patient01347_Plane1_19_of_24", "Axial", 1),
    ("Patient01365_Plane1_10_of_17", "Axial", 1),
    ("Patient01373_Plane1_7_of_9", "Axial", 1),
    ("Patient01408_Plane1_1_of_15", "Axial", 1),
    ("Patient01481_Plane1_33_of_35", "Axial", 0),
    ("Patient01486_Plane1_13_of_26", "Axial", 1),
    ("Patient01513_Plane1_14_of_14", "Axial", 0),
    ("Patient01517_Plane1_1_of_20", "Axial", 0),
    ("Patient01531_Plane1_1_of_1", "Axial", 1),
    ("Patient01566_Plane1_20_of_29", "Axial", 0),
    ("Patient01573_Plane1_4_of_8", "Axial", 0),
    ("Patient01583_Plane1_1_of_1", "Axial", 1),
    ("Patient01617_Plane1_1_of_1", "Axial", 1),
    ("Patient01617_Plane1_1_of_1", "Axial", 1),
    ("Patient01631_Plane1_2_of_16", "Axial", 1),
    ("Patient01635_Plane3_1_of_4", "Axial", 1),
    ("Patient01680_Plane1_1_of_1", "Axial", 1),
    ("Patient01719_Plane1_1_of_1", "Axial", 1),
    ("Patient01721_Plane1_1_of_1", "Axial", 1),
    ("Patient01736_Plane1_1_of_1", "Axial", 1),
    ("Patient01790_Plane1_1_of_1", "Axial", 1),
    ("Patient00764_Plane1_1_of_1", "Brain", 1),
    ("Patient00790_Plane1_19_of_47", "Brain", 1),
    ("Patient00813_Plane1_2_of_10", "Brain", 1),
    ("Patient00813_Plane1_8_of_10", "Brain", 1),
    ("Patient00854_Plane1_12_of_32", "Brain", 1),
    ("Patient00854_Plane1_2_of_32", "Brain", 1),
    ("Patient00855_Plane1_32_of_39", "Brain", 1),
    ("Patient00873_Plane1_14_of_23", "Brain", 1),
    ("Patient00875_Plane1_27_of_37", "Brain", 1),
    ("Patient00890_Plane1_5_of_19", "Brain", 1),
    ("Patient01169_Plane4_4_of_4", "Brain", 0),
    ("Patient01181_Plane5_3_of_3", "Brain", 1),
    ("Patient01188_Plane1_1_of_2", "Brain", 1),
    ("Patient01198_Plane1_7_of_9", "Brain", 0),
    ("Patient01212_Plane1_1_of_1", "Brain", 1),
    ("Patient01240_Plane1_1_of_1", "Brain", 1),
    ("Patient01240_Plane1_1_of_1", "Brain", 1),
    ("Patient01296_Plane1_1_of_1", "Brain", 1),
    ("Patient01391_Plane1_1_of_26", "Brain", 1),
    ("Patient01397_Plane1_3_of_9", "Brain", 1),
    ("Patient01484_Plane1_12_of_16", "Brain", 1),
    ("Patient01535_Plane1_18_of_21", "Brain", 1),
    ("Patient01535_Plane1_5_of_21", "Brain", 1),
    ("Patient01653_Plane4_1_of_6", "Brain", 0),
    ("Patient00722_Plane1_1_of_1", "Other", 1),
    ("Patient00789_Plane1_11_of_64", "Other", 1),
    ("Patient00799_Plane1_21_of_28", "Other", 1),
    ("Patient00889_Plane1_7_of_12", "Other", 0),
    ("Patient00908_Plane1_3_of_4", "Other", 1),
    ("Patient00961_Plane1_1_of_1", "Other", 1),
    ("Patient01036_Plane1_1_of_1", "Other", 1),
    ("Patient01149_Plane1_1_of_1", "Other", 1),
    ("Patient01287_Plane1_6_of_10", "Other", 1),
    ("Patient01367_Plane1_1_of_8", "Other", 1),
    ("Patient01473_Plane1_28_of_30", "Other", 1),
    ("Patient01513_Plane1_3_of_14", "Other", 0),
    ("Patient01547_Plane1_1_of_18", "Other", 1),
    ("Patient01572_Plane1_5_of_9", "Other", 1),
    # ("Patient00168_Plane1_20_of_23", "Mid-sagittal", 1),
    # ("Patient00168_Plane1_4_of_23", "Mid-sagittal", 1),
    # ("Patient00168_Plane1_7_of_23", "Mid-sagittal", 1),
    # ("Patient00188_Plane1_2_of_3", "Mid-sagittal", 1),
    # ("Patient00188_Plane1_3_of_3", "Mid-sagittal", 1),
    # ("Patient00684_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient00720_Plane1_10_of_27", "Mid-sagittal", 1),
    # ("Patient00739_Plane1_6_of_8", "Mid-sagittal", 1),
    # ("Patient00742_Plane1_15_of_20", "Mid-sagittal", 1),
    # ("Patient00768_Plane1_1_of_26", "Mid-sagittal", 1),
    # ("Patient00768_Plane1_22_of_26", "Mid-sagittal", 1),
    # ("Patient00768_Plane1_23_of_26", "Mid-sagittal", 1),
    # ("Patient00768_Plane1_3_of_26", "Mid-sagittal", 1),
    # ("Patient00769_Plane1_6_of_17", "Mid-sagittal", 1),
    # ("Patient00770_Plane1_1_of_2", "Mid-sagittal", 1),
    # ("Patient00771_Plane1_26_of_40", "Mid-sagittal", 1),
    # ("Patient00789_Plane1_8_of_64", "Mid-sagittal", 1),
    # ("Patient00790_Plane1_17_of_47", "Mid-sagittal", 1),
    # ("Patient00790_Plane1_43_of_47", "Mid-sagittal", 1),
    # ("Patient00790_Plane1_43_of_47", "Mid-sagittal", 1),
    # ("Patient00790_Plane1_44_of_47", "Mid-sagittal", 1),
    # ("Patient00791_Plane1_14_of_25", "Mid-sagittal", 1),
    # ("Patient00791_Plane1_20_of_25", "Mid-sagittal", 1),
    # ("Patient00791_Plane1_2_of_25", "Mid-sagittal", 1),
    # ("Patient00791_Plane1_9_of_25", "Mid-sagittal", 1),
    # ("Patient00792_Plane1_16_of_71", "Mid-sagittal", 1),
    # ("Patient00792_Plane1_29_of_71", "Mid-sagittal", 1),
    # ("Patient00792_Plane1_35_of_71", "Mid-sagittal", 1),
    # ("Patient00793_Plane1_10_of_48", "Mid-sagittal", 1),
    # ("Patient00793_Plane1_32_of_48", "Mid-sagittal", 1),
    # ("Patient00793_Plane1_33_of_48", "Mid-sagittal", 1),
    # ("Patient00795_Plane1_27_of_41", "Mid-sagittal", 1),
    # ("Patient00795_Plane1_35_of_41", "Mid-sagittal", 1),
    # ("Patient00795_Plane1_37_of_41", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_17_of_28", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_17_of_28", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_18_of_28", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_18_of_28", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_8_of_28", "Mid-sagittal", 1),
    # ("Patient00799_Plane1_8_of_28", "Mid-sagittal", 1),
    # ("Patient00809_Plane1_11_of_14", "Mid-sagittal", 1),
    # ("Patient00810_Plane1_3_of_11", "Mid-sagittal", 1),
    # ("Patient00813_Plane1_10_of_10", "Mid-sagittal", 1),
    # ("Patient00830_Plane1_18_of_49", "Mid-sagittal", 1),
    # ("Patient00830_Plane1_18_of_49", "Mid-sagittal", 1),
    # ("Patient00830_Plane1_2_of_49", "Mid-sagittal", 1),
    # ("Patient00831_Plane1_13_of_38", "Mid-sagittal", 1),
    # ("Patient00831_Plane1_26_of_38", "Mid-sagittal", 1),
    # ("Patient00831_Plane1_2_of_38", "Mid-sagittal", 1),
    # ("Patient00834_Plane1_13_of_34", "Mid-sagittal", 1),
    # ("Patient00835_Plane1_23_of_37", "Mid-sagittal", 1),
    # ("Patient00835_Plane1_8_of_37", "Mid-sagittal", 1),
    # ("Patient00853_Plane1_1_of_2", "Mid-sagittal", 1),
    # ("Patient00854_Plane1_24_of_32", "Mid-sagittal", 1),
    # ("Patient00854_Plane1_25_of_32", "Mid-sagittal", 1),
    # ("Patient00854_Plane1_28_of_32", "Mid-sagittal", 1),
    # ("Patient00855_Plane1_11_of_39", "Mid-sagittal", 1),
    # ("Patient00855_Plane1_16_of_39", "Mid-sagittal", 1),
    # ("Patient00855_Plane1_7_of_39", "Mid-sagittal", 1),
    # ("Patient00869_Plane1_2_of_5", "Mid-sagittal", 1),
    # ("Patient00870_Plane1_2_of_8", "Mid-sagittal", 1),
    # ("Patient00871_Plane1_4_of_11", "Mid-sagittal", 1),
    # ("Patient00871_Plane1_5_of_11", "Mid-sagittal", 1),
    # ("Patient00873_Plane1_11_of_23", "Mid-sagittal", 1),
    # ("Patient00873_Plane1_15_of_23", "Mid-sagittal", 1),
    # ("Patient00875_Plane1_2_of_37", "Mid-sagittal", 1),
    # ("Patient00875_Plane1_37_of_37", "Mid-sagittal", 1),
    # ("Patient00875_Plane1_37_of_37", "Mid-sagittal", 1),
    # ("Patient00889_Plane1_4_of_12", "Mid-sagittal", 1),
    # ("Patient00890_Plane1_4_of_19", "Mid-sagittal", 1),
    # ("Patient00893_Plane1_2_of_3", "Mid-sagittal", 1),
    # ("Patient00893_Plane1_2_of_3", "Mid-sagittal", 1),
    # ("Patient00909_Plane1_13_of_15", "Mid-sagittal", 1),
    # ("Patient00909_Plane1_14_of_15", "Mid-sagittal", 1),
    # ("Patient01032_Plane1_1_of_4", "Mid-sagittal", 1),
    # ("Patient01032_Plane1_2_of_4", "Mid-sagittal", 1),
    # ("Patient01120_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient01185_Plane1_11_of_15", "Mid-sagittal", 1),
    # ("Patient01185_Plane1_12_of_15", "Mid-sagittal", 1),
    # ("Patient01188_Plane1_2_of_2", "Mid-sagittal", 1),
    # ("Patient01198_Plane1_3_of_9", "Mid-sagittal", 1),
    # ("Patient01198_Plane1_4_of_9", "Mid-sagittal", 1),
    # ("Patient01198_Plane1_8_of_9", "Mid-sagittal", 1),
    # ("Patient01198_Plane1_9_of_9", "Mid-sagittal", 1),
    # ("Patient01199_Plane1_2_of_11", "Mid-sagittal", 1),
    # ("Patient01215_Plane1_6_of_24", "Mid-sagittal", 1),
    # ("Patient01215_Plane1_8_of_24", "Mid-sagittal", 1),
    # ("Patient01217_Plane1_1_of_5", "Mid-sagittal", 1),
    # ("Patient01217_Plane1_3_of_5", "Mid-sagittal", 1),
    # ("Patient01217_Plane1_4_of_5", "Mid-sagittal", 1),
    # ("Patient01218_Plane1_6_of_11", "Mid-sagittal", 1),
    # ("Patient01219_Plane1_1_of_2", "Mid-sagittal", 1),
    # ("Patient01221_Plane1_6_of_9", "Mid-sagittal", 1),
    # ("Patient01248_Plane1_5_of_15", "Mid-sagittal", 1),
    # ("Patient01249_Plane1_1_of_7", "Mid-sagittal", 1),
    # ("Patient01259_Plane1_2_of_7", "Mid-sagittal", 1),
    # ("Patient01272_Plane1_7_of_14", "Mid-sagittal", 1),
    # ("Patient01287_Plane1_5_of_10", "Mid-sagittal", 1),
    # ("Patient01287_Plane1_7_of_10", "Mid-sagittal", 1),
    # ("Patient01288_Plane1_1_of_4", "Mid-sagittal", 1),
    # ("Patient01307_Plane1_2_of_25", "Mid-sagittal", 1),
    # ("Patient01308_Plane1_8_of_9", "Mid-sagittal", 1),
    # ("Patient01327_Plane1_2_of_21", "Mid-sagittal", 1),
    # ("Patient01331_Plane1_1_of_4", "Mid-sagittal", 1),
    # ("Patient01332_Plane1_8_of_12", "Mid-sagittal", 1),
    # ("Patient01347_Plane1_13_of_24", "Mid-sagittal", 1),
    # ("Patient01347_Plane1_7_of_24", "Mid-sagittal", 1),
    # ("Patient01348_Plane1_5_of_8", "Mid-sagittal", 1),
    # ("Patient01364_Plane1_2_of_6", "Mid-sagittal", 1),
    # ("Patient01364_Plane1_3_of_6", "Mid-sagittal", 1),
    # ("Patient01365_Plane1_13_of_17", "Mid-sagittal", 1),
    # ("Patient01365_Plane1_16_of_17", "Mid-sagittal", 1),
    # ("Patient01367_Plane1_4_of_8", "Mid-sagittal", 1),
    # ("Patient01374_Plane1_1_of_12", "Mid-sagittal", 1),
    # ("Patient01374_Plane1_8_of_12", "Mid-sagittal", 1),
    # ("Patient01391_Plane1_23_of_26", "Mid-sagittal", 1),
    # ("Patient01395_Plane1_6_of_8", "Mid-sagittal", 1),
    # ("Patient01398_Plane1_1_of_10", "Mid-sagittal", 1),
    # ("Patient01398_Plane1_7_of_10", "Mid-sagittal", 1),
    # ("Patient01398_Plane1_8_of_10", "Mid-sagittal", 1),
    # ("Patient01402_Plane1_6_of_8", "Mid-sagittal", 1),
    # ("Patient01408_Plane1_5_of_15", "Mid-sagittal", 1),
    # ("Patient01408_Plane1_6_of_15", "Mid-sagittal", 1),
    # ("Patient01409_Plane1_19_of_27", "Mid-sagittal", 1),
    # ("Patient01409_Plane1_6_of_27", "Mid-sagittal", 1),
    # ("Patient01409_Plane1_7_of_27", "Mid-sagittal", 1),
    # ("Patient01422_Plane1_2_of_19", "Mid-sagittal", 1),
    # ("Patient01438_Plane1_14_of_15", "Mid-sagittal", 1),
    # ("Patient01442_Plane1_21_of_22", "Mid-sagittal", 1),
    # ("Patient01444_Plane1_11_of_28", "Mid-sagittal", 1),
    # ("Patient01444_Plane1_25_of_28", "Mid-sagittal", 1),
    # ("Patient01445_Plane1_1_of_11", "Mid-sagittal", 1),
    # ("Patient01446_Plane1_16_of_29", "Mid-sagittal", 1),
    # ("Patient01446_Plane1_17_of_29", "Mid-sagittal", 1),
    # ("Patient01448_Plane1_2_of_13", "Mid-sagittal", 1),
    # ("Patient01448_Plane1_3_of_13", "Mid-sagittal", 1),
    # ("Patient01448_Plane1_6_of_13", "Mid-sagittal", 1),
    # ("Patient01448_Plane1_9_of_13", "Mid-sagittal", 1),
    # ("Patient01450_Plane1_10_of_14", "Mid-sagittal", 1),
    # ("Patient01452_Plane1_3_of_9", "Mid-sagittal", 1),
    # ("Patient01452_Plane1_4_of_9", "Mid-sagittal", 1),
    # ("Patient01454_Plane1_2_of_6", "Mid-sagittal", 1),
    # ("Patient01454_Plane1_4_of_6", "Mid-sagittal", 1),
    # ("Patient01455_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient01469_Plane1_5_of_7", "Mid-sagittal", 1),
    # ("Patient01473_Plane1_15_of_30", "Mid-sagittal", 1),
    # ("Patient01473_Plane1_9_of_30", "Mid-sagittal", 1),
    # ("Patient01481_Plane1_17_of_35", "Mid-sagittal", 1),
    # ("Patient01481_Plane1_6_of_35", "Mid-sagittal", 1),
    # ("Patient01484_Plane1_8_of_16", "Mid-sagittal", 1),
    # ("Patient01485_Plane1_11_of_18", "Mid-sagittal", 1),
    # ("Patient01485_Plane1_2_of_18", "Mid-sagittal", 1),
    # ("Patient01485_Plane1_7_of_18", "Mid-sagittal", 1),
    # ("Patient01487_Plane1_17_of_17", "Mid-sagittal", 1),
    # ("Patient01487_Plane1_3_of_17", "Mid-sagittal", 1),
    # ("Patient01498_Plane1_1_of_3", "Mid-sagittal", 1),
    # ("Patient01500_Plane1_4_of_9", "Mid-sagittal", 1),
    # ("Patient01505_Plane1_12_of_24", "Mid-sagittal", 1),
    # ("Patient01505_Plane1_24_of_24", "Mid-sagittal", 1),
    # ("Patient01510_Plane1_2_of_12", "Mid-sagittal", 1),
    # ("Patient01513_Plane1_10_of_14", "Mid-sagittal", 1),
    # ("Patient01513_Plane1_6_of_14", "Mid-sagittal", 1),
    # ("Patient01513_Plane1_8_of_14", "Mid-sagittal", 1),
    # ("Patient01517_Plane1_11_of_20", "Mid-sagittal", 1),
    # ("Patient01517_Plane1_20_of_20", "Mid-sagittal", 1),
    # ("Patient01517_Plane1_5_of_20", "Mid-sagittal", 1),
    # ("Patient01535_Plane1_12_of_21", "Mid-sagittal", 1),
    # ("Patient01539_Plane1_14_of_25", "Mid-sagittal", 1),
    # ("Patient01545_Plane1_2_of_13", "Mid-sagittal", 1),
    # ("Patient01545_Plane1_3_of_13", "Mid-sagittal", 1),
    # ("Patient01547_Plane1_13_of_18", "Mid-sagittal", 1),
    # ("Patient01547_Plane1_5_of_18", "Mid-sagittal", 1),
    # ("Patient01548_Plane1_1_of_7", "Mid-sagittal", 1),
    # ("Patient01548_Plane1_7_of_7", "Mid-sagittal", 1),
    # ("Patient01549_Plane1_2_of_4", "Mid-sagittal", 1),
    # ("Patient01549_Plane1_4_of_4", "Mid-sagittal", 1),
    # ("Patient01551_Plane1_3_of_22", "Mid-sagittal", 1),
    # ("Patient01551_Plane1_8_of_22", "Mid-sagittal", 1),
    # ("Patient01566_Plane1_15_of_29", "Mid-sagittal", 1),
    # ("Patient01570_Plane1_15_of_16", "Mid-sagittal", 1),
    # ("Patient01570_Plane1_9_of_16", "Mid-sagittal", 1),
    # ("Patient01571_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient01572_Plane1_6_of_9", "Mid-sagittal", 1),
    # ("Patient01606_Plane1_5_of_18", "Mid-sagittal", 1),
    # ("Patient01612_Plane1_7_of_21", "Mid-sagittal", 1),
    # ("Patient01613_Plane1_10_of_15", "Mid-sagittal", 1),
    # ("Patient01654_Plane1_6_of_12", "Mid-sagittal", 1),
    # ("Patient01655_Plane1_7_of_12", "Mid-sagittal", 1),
    # ("Patient01655_Plane1_8_of_12", "Mid-sagittal", 1),
    # ("Patient01659_Plane1_4_of_6", "Mid-sagittal", 1),
    # ("Patient01671_Plane1_3_of_3", "Mid-sagittal", 1),
    # ("Patient01683_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient01698_Plane1_1_of_1", "Mid-sagittal", 1),
    # ("Patient01738_Plane1_2_of_2", "Mid-sagittal", 1),
]
for image_name, new_brain_plane, valid in remove_images:
    for index, row in fetal_planes[fetal_planes.Image_name == image_name].iterrows():
        fetal_planes.loc[index, "Valid"] = valid
        fetal_planes.loc[index, "New_brain_plane"] = new_brain_plane
        image = read_fetal_planes_image(image_name)
        images.append((image, f"{index}: {new_brain_plane}"))

        # print(f"\"{image_name}\"")
        # print(f"{index:<6}: {image_name} \t{row['Plane']}")
        # print(f"{index}, ", end="")


print(len(images))
if images:
    show_numpy_images(images[:36])

### Manual Inspection Helper

Single-image viewer: look up an image by its dataframe index, display it, and print its filename.
Used during manual curation to decide whether an image should be relabelled or invalidated.

In [ ]:
images = []

remove_images = [5899]
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    image = read_fetal_planes_image(image_name)
    images.append((image, f"{index}: {image_name}"))
    # print(f"\"{image_name}\"")
    print(f"{index:<6}: {image_name} \t{fetal_planes.Plane[index]}")
    # print(f"    (\"{image_name}\", \"Other\"),")

print()
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    print(f'"{image_name}"')

print()
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    print(f'    ("{image_name}", "Other"),')

print(len(images))
if images:
    show_numpy_images(images[:36])

### Statistics

Final label distribution: brain-plane counts for valid, non-duplicate images, including the `New_brain_plane` corrections.

In [ ]:
get_brain_planes_count(fetal_planes[fetal_planes["Duplicate"].isna()])

In [ ]:
df = fetal_planes[fetal_planes["Duplicate"].isna() & (fetal_planes["Valid"] == 1)].copy()
for index, row in df.iterrows():
    if isinstance(row["New_brain_plane"], str):
        df.loc[index, "Brain_plane"] = "Other"

get_brain_planes_count(df)

In [ ]:
# Same sanity check as the cell above (kept for re-running after manual label edits).
df = fetal_planes[fetal_planes["Duplicate"].isna() & (fetal_planes["Valid"] == 1)].copy()
for index, row in df.iterrows():
    if isinstance(row["New_brain_plane"], str):
        df.loc[index, "Brain_plane"] = "Other"

get_brain_planes_count(df)

### Save CSV

Final write of `data_fix.csv` with `Valid` and `New_brain_plane` columns fully populated.

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes